# qnn-edge-agent: Colab Runner (Day 1 E0 Baseline & Day 2 Ablation Suite)

Runs tests and executes full evaluation on GPU.

In [ ]:
# 1. GPU Check
!nvidia-smi

In [ ]:
# 2. Clone repository
import os
GIT_URL = "https://github.com/iamqazolp/qnn-edge-agent.git"  # Set git repo URL
REPO_NAME = "qnn-edge-agent"

if not os.path.exists(REPO_NAME):
    !git clone $GIT_URL $REPO_NAME
%cd $REPO_NAME

In [ ]:
# 3. Install dependencies in editable mode
!pip install -e ".[dev]"

In [ ]:
# 4. Run verification tests
!pytest tests/

In [ ]:
# 5. Run E0 evaluation (unconstrained and constrained)
!python scripts/run_e0.py --modes unconstrained,constrained

In [ ]:
# 6. Inspect E0 Results
import json
with open("results/E0.json") as f:
    res = json.load(f)

print(f"Model: {res['model_id']}")
print(f"WikiText-2 PPL: {res['wikitext2_ppl']}")
for mode, scores in res["eval"].items():
    print(f"\nMode: {mode}")
    for tier in ["json_valid", "tool_correct", "args_schema_ok", "values_exact"]:
        sc = scores[tier]
        print(f"  {tier:15s}: {sc['rate']*100:.1f}% [{sc['ci_low']*100:.1f}%, {sc['ci_high']*100:.1f}%] (n={sc['n']})")

# 7. Day 2 Quantization Ablation Suite (E0, E1, E2, E2a, E2b)

Runs the Day 2 quantization ablation suite across baseline FP32 (E0), plain INT8 (E1), naive W4A8 (E2), naive W4A16 (E2a), and FP32-W A8 (E2b).

In [ ]:
# 8. Run Day 2 ablation suite
!python scripts/run_ablation_suite.py --ids E0,E1,E2,E2a,E2b

In [ ]:
# 9. Summary comparison table
from pathlib import Path
import json

print(f"{'Recipe':<8} | {'Avg Bits':<8} | {'Footprint':<10} | {'PPL':<8} | {'Mode':<14} | {'JSON Valid':<11} | {'Exact Val':<11}")
print("-" * 80)
for rid in ["E0", "E1", "E2", "E2a", "E2b"]:
    p = Path(f"results/{rid}.json")
    if not p.exists():
        continue
    with open(p) as f:
        d = json.load(f)
    for mode, sc in d.get("eval", {}).items():
        ppl_str = f"{d.get('wikitext2_ppl', 0.0):.2f}" if d.get('wikitext2_ppl') else "N/A"
        print(f"{rid:<8} | {d['avg_bits_linear']:<8.1f} | {d['footprint_mb']:<7.1f} MB | {ppl_str:<8} | {mode:<14} | {sc['json_valid']['rate']*100:<10.1f}% | {sc['values_exact']['rate']*100:<10.1f}%")